# AIGC 5005 · Module 04 · Worked example
## Worked example: hardening the catalogue package

Companion notebook to the worked example of the Module 04 learning content. Every example from that part is here, ready to run, with comments explaining what to notice.

**Estimated time:** 45 minutes

**How to use it:** read each explanation, run the cell beneath it, and compare the output with what you expected *before* you ran it. The prediction is where the learning happens. Change values and run cells again; you cannot break anything.

**Kernel:** select the kernel from your `module04-quality` conda environment (top right in VS Code or Jupyter). To make one: `conda create -n module04-quality python=3.11 ipykernel`, then `conda activate module04-quality`. If VS Code offers to install `ipykernel`, accept.

**Tools:** this notebook runs `pytest`, `mypy` and `ruff` as programs. Install them into the same environment as the kernel with `pip install -r requirements.txt` (the file is in the top folder of this module's code). The notebook writes its examples into a scratch folder and delete it in the last cell.

## 0. Copy the project into a scratch folder

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

SOURCE = Path("catalogue_hardened")
SCRATCH = Path("scratch_worked").resolve()
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)
shutil.copytree(SOURCE, SCRATCH / "catalogue_hardened",
                ignore=shutil.ignore_patterns("__pycache__", "*.egg-info", ".*_cache"))
PROJECT = SCRATCH / "catalogue_hardened"

# The package lives in src/. Putting it on the path lets us run it without installing anything.
ENV = {**{k: v for k, v in os.environ.items() if k not in ("FORCE_COLOR", "CLICOLOR_FORCE")},
       "PYTHONPATH": str(PROJECT / "src"), "NO_COLOR": "1"}       # NO_COLOR: plain text, no colour codes


def run(*args, env=ENV):
    """Run `python <args>` in the project folder and show the output, as a terminal would."""
    result = subprocess.run([sys.executable, *args], cwd=PROJECT, capture_output=True, text=True, env=env)
    print("$ python", " ".join(args))
    print((result.stdout + result.stderr).rstrip() or "(no output)")
    print(f"[exit code {result.returncode}]")
    return result


print("Project copied to:", PROJECT)
for path in sorted(PROJECT.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(PROJECT).as_posix())

This is the Module 03 catalogue project after hardening. Compare it with `before_hardening/`, which is where Module 03 left it. Each step below matches a part of the module.

## 1. The four checks

Run them in the order from Part 4: format, lint, types, tests.

In [ ]:
run("-m", "ruff", "format", "--check", ".")
run("-m", "ruff", "check", ".")

In [ ]:
run("-m", "mypy")

In [ ]:
run("-m", "pytest", "-v")

## 2. The program

The program reads its settings from `config/catalogue.toml`. Run it from the project root, as the `README` says.

**Predict first:** three books are printed. Will the same three appear if you run the cell twice?

In [ ]:
print((PROJECT / "config" / "catalogue.toml").read_text())
run("-m", "catalogue.main")
run("-m", "catalogue.main")

The log lines and the book lines are different things: the log is diagnostics (on standard error), and the book list is the program's output. The seed makes the choice repeatable.

### What this run recorded

The program saved the settings and the environment next to its results.

In [ ]:
print((PROJECT / "data" / "processed" / "run_info.json").read_text())

## 3. Change the seed

Settings change without touching code.

In [ ]:
custom = PROJECT / "config" / "other_seed.toml"
custom.write_text((PROJECT / "config" / "catalogue.toml").read_text().replace("seed = 42", "seed = 99"))
run("-m", "catalogue.main", "--config", "config/other_seed.toml")

## 4. Make it fail usefully

A missing settings file, a missing catalogue and a broken catalogue should each produce a clear message and a non-zero exit code, never a traceback.

In [ ]:
run("-m", "catalogue.main", "--config", "config/nothing.toml")

In [ ]:
(PROJECT / "data" / "catalogue.json").write_text("[{", encoding="utf-8")
run("-m", "catalogue.main")

Each failure names what went wrong and where. The exit code of 1 lets another program, or a continuous integration service, tell that the run failed.

## 5. Break a test on purpose

Change the expected answer in a test and watch the report. Then restore it.

In [ ]:
test_file = PROJECT / "tests" / "test_models.py"
original = test_file.read_text()
test_file.write_text(original.replace('("REFACTORING", "Refactoring")', '("REFACTORING", "REFACTORING")'))
run("-m", "pytest", "tests/test_models.py", "-q")
test_file.write_text(original)

---
## Try it

Add a `Book.renew()` method that adds one copy and logs it. Write the tests first, including one for a book that has been returned twice. Add type hints. Run all four checks.

## Clean up

Removes the scratch folder.

In [ ]:
shutil.rmtree(SCRATCH)
print("Scratch folder removed:", not SCRATCH.exists())